# EmpowerLens — Re-annotation triage Kaggle GPU runner

Runs Stage 1 (K-fold out-of-fold prediction) and Stage 2 (triage scoring) of
model-assisted re-annotation on Kaggle's free GPU, then makes
`results/reannotation/` downloadable — in particular `audit_queue.csv`,
which is the file to send out for manual review.

**Before running:**
1. Settings -> **Accelerator: GPU**, and **Internet: On** (needed for `pip` +
   the `mental/mental-roberta-base` download from the Hugging Face Hub).
2. This notebook does **not** need `data/splits/` — it re-annotates the
   whole corpus via K-fold CV, independent of the frozen train/val/test
   split used for benchmark training.
3. This is a data-quality tool, not a benchmark run: epochs are kept low
   (default 3) and there is no 3-seed averaging — only the out-of-fold
   probability matrix matters here, not a reportable model.

This notebook only orchestrates shell commands; all logic lives in `src/`.

In [ ]:
# 1. Clone the repo and install the transformer stack + re-annotation extras.
REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "lumia-space"

!rm -rf empowerlens && git clone --branch $BRANCH $REPO_URL empowerlens
%cd empowerlens
!pip install -q -r requirements-transformer.txt
!pip install -q cleanlab iterative-stratification

In [ ]:
# 2. Sanity check: confirm GPU is actually attached before a multi-fold run.
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (turn on the GPU accelerator!)")

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token)


In [ ]:
# 3. Stage 1 — K-fold out-of-fold prediction across the whole corpus.
#    --device auto resolves to the Kaggle GPU (cuda). Single-GPU pinning
#    (matches the deadlock fix from the benchmark runner) in case the
#    session has 2x T4s.
#
#    MAX_POS_WEIGHT: a first run at the old default (10.0) trained cleanly
#    (inner-val macro_f1 climbed to 0.23-0.28 and plateaued across all 5
#    folds — not degenerate) but still over-predicted ~3x (avg 2.54
#    positives/row vs a ~0.8 true corpus average). That's pos_weight
#    pushing probabilities up corpus-wide, not an undertraining problem, so
#    the fix is a lower cap here, not more epochs. 3.0 is a starting point —
#    after this cell finishes, check the per-class breakdown it prints (and
#    results/reannotation/oof_per_class_stats.csv): if avg positives/row is
#    still well above ~1.5, drop to 2.0 and re-run; if per-class prob std
#    collapses below ~0.15 for several classes, that's this cap being too
#    aggressive going the other way.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

FOLDS = 5
EPOCHS = 5
SEED = 42
MAX_POS_WEIGHT = 3.0

!python -m src.reannotate_oof_predict \
    --folds $FOLDS --epochs $EPOCHS --seed $SEED --device auto \
    --max-pos-weight $MAX_POS_WEIGHT \
    --out results/reannotation

In [ ]:
# 4. Stage 2 — score + bucket every row, build the priority-sorted audit queue.
#    Adjust --queue-size / --conf-low / --entropy-quantile here if the
    # bucket counts printed below look off (e.g. bucket A/B way over 500 rows,
    # tighten --conf-low; way under 300, loosen it) — but check
    # results/reannotation/triage_per_class_stats.csv and the printed
    # per-class breakdown first: if bucket A is inflated because Stage 1 is
    # still over-predicting corpus-wide, re-run cell 3 with a lower
    # MAX_POS_WEIGHT instead of loosening --conf-low here, which would mask
    # rather than fix the calibration issue.
QUEUE_SIZE = 400

!python -m src.reannotate_triage \
    --oof-dir results/reannotation --out results/reannotation \
    --queue-size $QUEUE_SIZE

In [ ]:
# 5. Copy results to the Kaggle output so it can be downloaded from the session.
#    audit_queue.csv is the one to open in Excel/Sheets and pass around for
    # manual review (corrected_primary / corrected_secondary / reviewer / review_notes).
!mkdir -p /kaggle/working/results/reannotation
!cp -r results/reannotation/* /kaggle/working/results/reannotation/
!ls -la /kaggle/working/results/reannotation

## Next step (off-Kaggle)

1. Download `results/reannotation/audit_queue.csv` from the Kaggle session
   output (right panel -> Output -> `results/reannotation/`).
2. Circulate it to Nayab / Izza (and Lumia) — fill in `corrected_primary`
   and `corrected_secondary` for each row, `reviewer`, and any `review_notes`.
3. Once completed, run locally (or in a fresh Kaggle session with the
   completed CSV re-uploaded):

   ```
   python -m src.reannotate_merge_audit \
       --audit results/reannotation/audit_queue.csv \
       --out data/Annotated_data_v2.csv
   ```

   This produces the versioned, provenance-tracked relabeled dataset.
   `Annotated_data.csv` itself is never modified.